# 고지방식 vs 고전분식 마우스 간 단백체: 후분석 실습

**Bertis 단백체 데이터 분석 워크샵 · Hands-on**

limma로 미리 계산한 DE 결과에서 시작해 다음 순서로 진행합니다.

1. DE 결과 살펴보기 (volcano plot)
2. ORA: g:Profiler
3. GSEA: gseapy prerank (GO Biological Process)
4. 단백질 상호작용 네트워크: STRING
5. 논문 결론과 연결: 일반식 대비 두 식이
6. (부록) 원자료 둘러보기: 결측, 정규화, PCA

**데이터**: Small et al., *J Lipid Res* 2024 (PXD054323). C57BL/6J 수컷 마우스, 4주, 간 조직, label-free DDA (MaxLFQ).
비교: 고지방식(HFD, n = 14) vs 고전분식(HSD, n = 13). 근육 조직이 섞인 샘플은 제외.

**실행 방법**: 코드를 고칠 필요 없이 위에서부터 셀을 하나씩 실행하세요 (`Shift + Enter`). 결과를 남기려면 `파일 > 드라이브에 사본 저장`.

## 0. 준비

In [ ]:
# 패키지 설치 (Colab 기준 30초 내외)
# 그림의 글자는 Colab에 한글 글꼴이 없어 영어로 표시합니다
!pip install -q gseapy gprofiler-official

In [ ]:
import os, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import requests
import gseapy as gp
from gprofiler import GProfiler

# 실습 자료 저장소 (raw 주소)
REPO_RAW = "https://raw.githubusercontent.com/ORG/REPO/main"   # 배포 전에 실제 저장소로 교체

def load_file(rel_path):
    # 노트북 옆에 파일이 있으면 그것을, 없으면 저장소에서 읽어 옵니다
    if os.path.exists(rel_path):
        return rel_path
    r = requests.get(f"{REPO_RAW}/{rel_path}", timeout=60)
    r.raise_for_status()
    os.makedirs(os.path.dirname(rel_path) or ".", exist_ok=True)
    open(rel_path, "wb").write(r.content)
    return rel_path

UP, DOWN, NS, NAVY = "#BF6A8B", "#003399", "#C7CFDA", "#0B1F3A"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. DE 결과 살펴보기

limma로 계산한 결과표입니다. `log2FC`는 HFD / HSD 비율의 log2 값이라, **음수면 고지방식에서 줄어든 단백질**입니다.

In [ ]:
de = pd.read_csv(load_file("data/de_results_limma.csv"))
print(f"검정한 단백질: {len(de):,}개")
de.head(10)

선택 기준을 정합니다. 기준을 바꿔 가며 개수가 어떻게 달라지는지도 확인해 보세요.

In [ ]:
FDR_CUT = 0.05          # adj. p 기준
LFC_CUT = np.log2(1.5)  # 1.5배 이상 변화

de["direction"] = "NS"
de.loc[(de["adj.P.Val"] < FDR_CUT) & (de["log2FC"] >= LFC_CUT), "direction"] = "Up"
de.loc[(de["adj.P.Val"] < FDR_CUT) & (de["log2FC"] <= -LFC_CUT), "direction"] = "Down"
de["direction"].value_counts()

In [ ]:
def volcano(df, label_genes=("Fasn", "Acly", "Acaca", "Scd1", "Me1", "Ech1", "Cyp4a10", "Vnn1")):
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(df["log2FC"], -np.log10(df["P.Value"]), c=df["direction"].map({"Up": UP, "Down": DOWN, "NS": NS}),
               s=10, lw=0, alpha=0.9)
    ax.axvline(LFC_CUT, ls="--", c="grey", lw=0.8); ax.axvline(-LFC_CUT, ls="--", c="grey", lw=0.8)
    for _, r in df[df["gene"].isin(label_genes)].iterrows():
        ax.annotate(r["gene"], (r["log2FC"], -np.log10(r["P.Value"])), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("log2 fold change (HFD / HSD)"); ax.set_ylabel("-log10 p-value")
    ax.set_title(f"Down {sum(df.direction=='Down')} · Up {sum(df.direction=='Up')}")
    plt.show()

volcano(de)

## 2. ORA: g:Profiler

목록 안에 어떤 기능군이 우연보다 많이 들어 있는지 봅니다. 웹의 [g:Profiler](https://biit.cs.ut.ee/gprofiler/gost)와 같은 서버를 씁니다.
배경(background)은 **이 실험에서 검출된 단백질 전체**로 지정합니다.

먼저 고지방식에서 **감소한** 단백질부터 봅니다.

In [ ]:
down_genes = de.loc[de.direction == "Down", "gene"].dropna().unique().tolist()
up_genes = de.loc[de.direction == "Up", "gene"].dropna().unique().tolist()
background = de["gene"].dropna().unique().tolist()
print(f"Down {len(down_genes)}개, Up {len(up_genes)}개, 배경 {len(background)}개")

def run_ora(genes, label):
    try:
        gpr = GProfiler(return_dataframe=True, user_agent="bertis_workshop")
        return gpr.profile(organism="mmusculus", query=genes, background=background, domain_scope="custom",
                           sources=["GO:BP", "KEGG", "REAC"], user_threshold=0.05)
    except Exception as e:
        print(f"g:Profiler 접속 실패 ({type(e).__name__}), 저장된 결과를 불러옵니다.")
        return pd.read_csv(load_file(f"data/backup/ora_{label}.csv"))

def dotplot(res, title, n=12):
    if len(res) == 0:
        print("유의한 term이 없습니다."); return
    d = res.nsmallest(n, "p_value").iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.38 * len(d) + 1))
    sc = ax.scatter(-np.log10(d["p_value"]), d["name"].str.slice(0, 55), s=d["intersection_size"] * 12,
                    c=d["intersection_size"] / d["term_size"], cmap="Blues", edgecolor=NAVY, lw=0.5)
    ax.set_xlabel("-log10 adjusted p-value"); ax.set_title(title)
    plt.colorbar(sc, label="gene ratio (overlap / term size)")
    plt.tight_layout(); plt.show()

ora_down = run_ora(down_genes, "down")
dotplot(ora_down, "ORA: down in HFD")
ora_down[["source", "name", "p_value", "term_size", "intersection_size"]].head(10)

증가한 단백질로도 해 보세요. 목록이 짧으면 ORA 결과가 약하게 나올 수 있는데, 이런 경우 다음 단계의 GSEA가 유리합니다.

In [ ]:
ora_up = run_ora(up_genes, "up")
dotplot(ora_up, "ORA: up in HFD")

## 3. GSEA: gseapy prerank (GO Biological Process)

목록을 자르지 않고, 검정한 모든 단백질을 limma의 t 통계량 순서로 세운 뒤 각 gene set이 위쪽(HFD에서 증가) 또는 아래쪽(HSD에서 증가)에 몰려 있는지 봅니다.

- gene set: GO Biological Process (MSigDB C5 GO:BP v7.5.1, 마우스 ortholog). 저장소에 파일로 넣어 두어 모두 같은 결과가 나옵니다.
- gene set 크기: 100 ~ 500개 (논문과 같은 기준). 너무 작은 term은 빼고 큰 흐름을 봅니다.

In [ ]:
rnk = (de.dropna(subset=["gene"]).assign(a=lambda d: d.t.abs()).sort_values("a", ascending=False)
         .drop_duplicates("gene").set_index("gene")["t"].sort_values(ascending=False))

try:
    gobp = gp.read_gmt(load_file("data/genesets/GO_BP_mouse.gmt"))
except Exception:
    print("저장소 gene set을 읽지 못해 Enrichr에서 받아옵니다.")
    gobp = gp.get_library("GO_Biological_Process_2023", organism="Mouse")
print(f"gene set {len(gobp)}개")

pre = gp.prerank(rnk=rnk, gene_sets=gobp, min_size=100, max_size=500,
                 permutation_num=1000, seed=7, threads=2, outdir=None, verbose=False)
gsea = pre.res2d.copy()
gsea["NES"] = gsea["NES"].astype(float); gsea["FDR q-val"] = gsea["FDR q-val"].astype(float)
gsea = gsea.sort_values("NES", ascending=False)
print("검정한 term:", len(gsea), "| FDR < 0.05:", (gsea["FDR q-val"] < 0.05).sum())
print("HFD에서 증가 쪽 (NES > 0)")
gsea[["Term", "NES", "FDR q-val", "Tag %"]].head(8)

In [ ]:
print("HSD에서 증가 쪽 (NES < 0)")
gsea.sort_values("NES")[["Term", "NES", "FDR q-val", "Tag %"]].head(8)

In [ ]:
# 논문 Fig. 4A와 비슷한 형식: 방향별 상위 6개 + lipid biosynthetic process
key = gsea[gsea.Term.str.startswith("lipid biosynthetic process (")]
top = pd.concat([gsea[gsea.NES > 0].head(6), gsea[gsea.NES < 0].tail(6), key]).drop_duplicates("Term")
top = top.assign(label=top.Term.str.replace(r" \(GO:\d+\)", "", regex=True)).sort_values("NES")
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(top.label, top.NES, color=np.where(top.NES > 0, UP, DOWN))
for y, (n, q) in enumerate(zip(top.NES, top["FDR q-val"])):
    ax.text(n + (0.05 if n > 0 else -0.05), y, f"q={q:.3f}", va="center", ha="left" if n > 0 else "right", fontsize=8)
ax.axvline(0, c="grey", lw=0.8); ax.set_xlim(-3.3, 3.3); ax.set_xlabel("NES")
ax.set_title("GSEA GO-BP  (pink: up in HFD / blue: up in HSD)"); plt.tight_layout(); plt.show()

In [ ]:
# 어떤 단백질이 신호를 만들었는지: leading edge
for term in ["small molecule catabolic process", "lipid biosynthetic process"]:
    row = gsea[gsea.Term.str.startswith(term + " (")].iloc[0]
    print(f"{term}: NES {row.NES:.2f}, FDR {row['FDR q-val']:.3f}")
    print("  leading edge:", ", ".join(row.Lead_genes.split(";")[:12]), "\n")

In [ ]:
# enrichment plot: 증가 쪽 1위와 지방 합성
for term in [gsea.iloc[0]["Term"], gsea[gsea.Term.str.startswith("lipid biosynthetic process (")].iloc[0]["Term"]]:
    pre.plot(terms=term, figsize=(5.5, 4.5)); plt.show()

## 4. 단백질 상호작용 네트워크: STRING

고지방식에서 감소한 단백질 사이의 알려진 상호작용을 STRING에서 받아 그립니다. 신뢰도 점수 0.7(high) 이상만 씁니다.

In [ ]:
def string_network(genes, label, score=700):
    try:
        r = requests.post("https://string-db.org/api/tsv/network",
                          data={"identifiers": "\r".join(genes), "species": 10090,
                                "required_score": score, "caller_identity": "bertis_workshop"}, timeout=60)
        r.raise_for_status()
        net = pd.read_csv(io.StringIO(r.text), sep="\t")
    except Exception as e:
        print(f"STRING 접속 실패 ({type(e).__name__}), 저장된 결과를 불러옵니다.")
        net = pd.read_csv(load_file(f"data/backup/string_{label}.tsv"), sep="\t")
    return net.drop_duplicates(["preferredName_A", "preferredName_B"])

net = string_network(down_genes, "down")
G = nx.from_pandas_edgelist(net, "preferredName_A", "preferredName_B", edge_attr="score")
deg = dict(G.degree())
print(f"단백질 {G.number_of_nodes()}개, 연결 {G.number_of_edges()}개")

fig, ax = plt.subplots(figsize=(8, 7))
pos = nx.spring_layout(G, seed=3, k=0.5)
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.35, width=[0.5 + 2 * (d["score"] - 0.7) / 0.3 for *_, d in G.edges(data=True)])
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=[60 + deg[n] * 40 for n in G], node_color=DOWN, edgecolors="white", linewidths=0.8)
nx.draw_networkx_labels(G, pos, font_size=9, font_color=NAVY, ax=ax)
ax.set_title("STRING network (down in HFD, score ≥ 0.7)"); ax.axis("off"); plt.show()
pd.Series(deg).sort_values(ascending=False).head(10).rename("degree")

In [ ]:
# 같은 목록을 STRING 웹사이트에서 열어 보기 (출력된 주소를 클릭)
print("https://string-db.org/cgi/network?species=10090&identifiers=" + "%0d".join(down_genes))

## 5. 논문 결론과 연결: 일반식 대비 두 식이

같은 방식으로 **고전분식 vs 일반식**, **고지방식 vs 일반식**을 분석한 결과표를 겹쳐 봅니다.
간 중성지방은 두 식이 모두 비슷하게 늘었지만(샘플 정보의 `liver_triglyceride_content`), 지방 합성률(`liver_lipogenesis`)은 정반대입니다.

In [ ]:
samples = pd.read_csv(load_file("data/samples.csv"))
s = samples[~samples.muscle_contaminated]
print(s.groupby("group")[["liver_triglyceride_content", "liver_lipogenesis"]].mean().round(1))

a = pd.read_csv(load_file("data/de_HFD_vs_Chow.csv"))
b = pd.read_csv(load_file("data/de_HSD_vs_Chow.csv"))
m = a.merge(b, on=["protein", "gene"], suffixes=("_HFD", "_HSD"))
lip = ["Fasn", "Acly", "Acaca", "Acacb", "Scd1", "Me1", "Acss2", "G6pdx"]
tg = ["Plin2", "Apoa4", "Fabp2"]
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(m.log2FC_HFD, m.log2FC_HSD, s=6, c=NS)
for genes_, c in [(lip, DOWN), (tg, UP)]:
    d = m[m.gene.isin(genes_)]
    ax.scatter(d.log2FC_HFD, d.log2FC_HSD, s=40, c=c, edgecolor="white", zorder=3)
    for _, r in d.iterrows():
        ax.annotate(r.gene, (r.log2FC_HFD, r.log2FC_HSD), fontsize=8, xytext=(3, 3), textcoords="offset points", color=c)
ax.axhline(0, c="grey", lw=0.7); ax.axvline(0, c="grey", lw=0.7)
ax.set_xlim(-2.3, 2.3); ax.set_ylim(-1.6, 1.9)
ax.set_xlabel("log2FC  HFD vs Chow"); ax.set_ylabel("log2FC  HSD vs Chow")
ax.set_title("lipogenesis: opposite / lipid droplet: both up"); plt.show()

## 정리

- **HSD에서 증가**: 지방 합성 효소(Acaca, Acly, Fasn, Scd1, Me1, G6pdx), GO-BP *lipid biosynthetic process* · *carbohydrate metabolic process*
- **HFD에서 증가**: 지방산 산화 · 이화 관련 단백질(Ech1, Eci1, Cyp4a10 등), GO-BP *small molecule catabolic process* · *lipid catabolic process*

탄수화물이 많은 식이는 간의 지방 합성을 켜고, 지방이 많은 식이는 이를 끄고 지방을 태우는 쪽으로 바꾼다는 결과입니다.
일반식과 비교하면 간 지방(TG)은 두 식이에서 비슷하게 쌓이지만 지방 합성은 정반대로 움직입니다. 즉 **간 지방 축적과 지방 합성은 따로 움직인다**는 것이 Small et al. (2024)의 결론입니다.

- ORA와 GSEA에서 공통으로 나온 경로는 무엇인가요?
- 증가 쪽은 ORA보다 GSEA에서 더 잘 보이나요? 이유는 무엇일까요?
- STRING 모듈이 ORA · GSEA 결과와 맞나요?

## 부록: 원자료 둘러보기

DE 결과표가 만들어지기 전의 단백질 × 샘플 행렬(4주, 세 식이 45개 샘플)입니다. 슬라이드의 전처리 · QC 그림을 직접 그려 볼 수 있습니다.

In [ ]:
raw = pd.read_csv(load_file("data/protein_matrix_raw_log2.csv"), index_col=0)    # log2 LFQ, 정규화 전
print(raw.shape, f"결측 {raw.isna().mean().mean()*100:.1f}%")
samples.groupby("group").size()

In [ ]:
# 신호가 약할수록 결측이 많다 (MNAR)
m = raw.mean(axis=1); miss = raw.isna().mean(axis=1) * 100
pd.Series(miss.values, index=pd.cut(m, np.arange(18, 40, 1))).groupby(level=0, observed=True).mean().plot.bar(color=UP, figsize=(7, 3))
plt.xlabel("mean log2 intensity"); plt.ylabel("missing (%)"); plt.tight_layout(); plt.show()

In [ ]:
# 정규화 전후 분포
norm = raw - raw.median() + raw.median().median()
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
raw.boxplot(ax=axs[0], showfliers=False, grid=False); axs[0].set_title("before")
norm.boxplot(ax=axs[1], showfliers=False, grid=False); axs[1].set_title("after median normalization")
for a in axs: a.set_xticks([])
plt.show()

In [ ]:
# PCA: 근육이 섞인 샘플이 어디에 있는지 보세요
X = norm.dropna()
Z = (X.sub(X.mean(axis=1), axis=0)).T.values
U, s, Vt = np.linalg.svd(Z - Z.mean(0), full_matrices=False)
pc = U[:, :2] * s[:2]
colors = {"HFD": UP, "HSD": "#1FA7E1", "Chow": "#8A94A6"}
fig, ax = plt.subplots(figsize=(6, 5))
for g, c in colors.items():
    k = (samples.group == g).values
    ax.scatter(pc[k, 0], pc[k, 1], c=c, label=g, s=35, edgecolor="white")
k = samples.muscle_contaminated.values
ax.scatter(pc[k, 0], pc[k, 1], s=120, facecolor="none", edgecolor=NAVY, label="muscle contaminated")
ax.legend(frameon=False); ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); plt.show()